# Topics Comparison

Notebook for comparing `S3`, `FASTopic`, and a `TF-IDF` baseline on canonical Polymarket markets. The default text view is `full_description`, but every detector also supports `question` and `question_plus_full_description`.

In [1]:
from polymarket_research.data.canonical import CanonicalDatasetBuilder
from polymarket_research.data.raw import RawExternalCovariates, RawMarketHandle
from polymarket_research.utils import setup_root

from polymarket_research.research.topic_models import (
    FASTopicFactory,
    S3TopicFactory,
    TFIDFTopicFactory,
    build_topic_input_frame,
    compare_topic_factories,
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', None)
%matplotlib inline


/Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/_snmf.py:19: UserWarning: JAX not found, continuing with NumPy implementation.
  warnings.warn("JAX not found, continuing with NumPy implementation.")


In [2]:
REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'
MARKET_LIMIT = None
MARKET_ORDER = None  # one of: None, "latest", "largest"

if (CANONICAL_CACHE_DIR / 'markets.parquet').exists() and MARKET_LIMIT is None and MARKET_ORDER is None:
    from polymarket_research.data.canonical import CanonicalDataset
    canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
    print('Loaded canonical from parquet cache:', CANONICAL_CACHE_DIR)
else:
    raw_handle = RawMarketHandle(source=DATA_SOURCE)
    raw_bundle = raw_handle.load_bundle(
        include_market_universe=False,
        include_download_manifest=False,
        include_probabilities=True,
        include_raw_trades=False,
        market_limit=MARKET_LIMIT,
        market_order=MARKET_ORDER,
    )
    raw_external = RawExternalCovariates().load()
    canonical = CanonicalDatasetBuilder(
        raw_dataset=raw_bundle,
        raw_external=raw_external,
        resolved_only=True,
    ).build()
    if MARKET_LIMIT is None and MARKET_ORDER is None:
        canonical.save(CANONICAL_CACHE_DIR)
    print('Built canonical from SQLite')

all_markets = canonical.markets.copy()
probabilities = canonical.probabilities.copy()
print(f'Canonical resolved markets: {len(all_markets)}')
print('Data source:', DATA_SOURCE)
print('Market limit:', MARKET_LIMIT)
print('Market order:', MARKET_ORDER)
display(all_markets[["market_id", "question", "research_category", "family_id", "volume_num"]].head(5))


KeyboardInterrupt: 

In [ ]:
topic_markets = build_topic_input_frame(all_markets)
topic_markets["question_len"] = topic_markets["question"].fillna("").str.len()
topic_markets["full_description_len"] = topic_markets["full_description"].fillna("").str.len()

coverage = pd.DataFrame({
    "n_rows": [len(topic_markets)],
    "nonempty_question": [int(topic_markets["question"].fillna("").str.strip().ne("").sum())],
    "nonempty_full_description": [int(topic_markets["full_description"].fillna("").str.strip().ne("").sum())],
    "mean_question_len": [float(topic_markets["question_len"].mean())],
    "mean_full_description_len": [float(topic_markets["full_description_len"].mean())],
})
display(coverage)
display(topic_markets[["market_id", "question", "full_description", "volume_num"]].sample(2))

## Run Config

Use `TEXT_MODE = "full_description"` for the description-centric view. Switch to `"question"` or `"question_plus_full_description"` to compare text views.

In [ ]:
TEXT_MODE = "full_description"
N_TOPICS = 20
MIN_DF = 10
MAX_DF = 0.4
TOP_TERMS = 8
FASTOPIC_EPOCHS = 100
SAMPLE_SIZE = None
SORT_BY = "volume_num"
REDUCERS = ["umap", "tsne"]

work_markets = topic_markets.copy()
work_markets = work_markets.loc[work_markets[TEXT_MODE].fillna("").astype(str).str.strip().ne("")].copy()
if SORT_BY in work_markets.columns:
    work_markets = work_markets.sort_values(SORT_BY, ascending=False, kind="stable")
if SAMPLE_SIZE is not None:
    work_markets = work_markets.head(int(SAMPLE_SIZE)).copy()

print("Rows used for modeling:", len(work_markets))
display(work_markets[["market_id", "question", "full_description", "volume_num"]].sample(2))

In [ ]:
topic_factories = [
    S3TopicFactory(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
    ),
    FASTopicFactory(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
        n_epochs=FASTOPIC_EPOCHS,
    ),
    TFIDFTopicFactory(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
    ),
]

results, comparison_summary = compare_topic_factories(work_markets, topic_factories)
display(comparison_summary)

In [ ]:
for model_name, result in results.items():
    print("=" * 100)
    print(model_name)
    display(result.topic_summary()[["topic_id", "topic_label", "topic_size"]])

In [ ]:
for model_name, result in results.items():
    fig, axes, projected = result.plot_2d_with_topic_map(
        reducer="umap",
        random_state=0,
        legend_mode="full",
        show_table=False,
        figsize=(18, 10),
    )
    plt.show()

In [ ]:
for model_name, result in results.items():
    fig, axes, projected = result.plot_2d_with_topic_map(
        reducer="tsne",
        random_state=0,
        legend_mode="full",
        show_table=False,
        figsize=(18, 10),
    )
    plt.show()

In [ ]:
TOP_DOCS_PER_TOPIC = 2
s3_model = results["S3"]

print("=" * 100)
print("Representative documents for S3")
display(
    s3_model.representative_documents(top_n=TOP_DOCS_PER_TOPIC)[
        ["market_id", "topic_id", "topic_confidence", "topic_label", "question", "full_description"]
    ]
)


## Enrichment With The Fitted S3 Model

Once the S3 model is fitted, use it as a reusable enrichment artifact. 
Apply it either to new market-shaped rows or to raw free-text descriptions.


In [ ]:
# The fitted S3 artifact can enrich either market rows or raw descriptions.
s3_model = results["S3"]

# 1. Enrich market-shaped rows.
example_new_markets = work_markets[["market_id", "question", "full_description"]].head(5).copy()
display(
    s3_model.infer_markets(example_new_markets)[
        ["market_id", "topic_id", "topic_confidence", "topic_label", "topic_terms", "question"]
    ]
)

# 2. Enrich raw descriptions directly.
example_new_descriptions = pd.Series([
    "Will the Fed cut rates by the September 2026 meeting?",
    "Will Nvidia close above $200 before the end of June 2026?",
    "Will Bitcoin trade above $150k in 2026?",
], name="description")

display(
    s3_model.infer_descriptions(example_new_descriptions)[
        ["input_index", "text_used", "topic_id", "topic_confidence", "topic_label", "topic_terms"]
    ]
)


## Why S3 Is The Working Default

We currently treat **S3** as the working default because it gave the best visual topic separation in this notebook. 
The cells below make that choice explicit in a paper-friendly way: a compact quantitative summary plus a small manual-review table.


In [ ]:
comparison_plot = comparison_summary.copy()
comparison_plot = comparison_plot.sort_values("runtime_seconds").reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=comparison_plot, x="model_name", y="mean_topic_confidence", ax=axes[0], palette="deep")
axes[0].set_title("Mean Topic Confidence")
axes[0].set_xlabel("")
axes[0].set_ylabel("mean_topic_confidence")

sns.barplot(data=comparison_plot, x="model_name", y="runtime_seconds", ax=axes[1], palette="deep")
axes[1].set_title("Runtime")
axes[1].set_xlabel("")
axes[1].set_ylabel("runtime_seconds")

fig.suptitle("Topic-model comparison summary")
fig.tight_layout()


In [ ]:
model_review_notes = pd.DataFrame([
    {
        "model_name": "S3",
        "visual_separation": "best",
        "topic_coherence_note": "working default after visual review",
        "decision": "selected",
    },
    {
        "model_name": "FASTopic",
        "visual_separation": "review",
        "topic_coherence_note": "keep as strong neural baseline",
        "decision": "baseline",
    },
    {
        "model_name": "TFIDF",
        "visual_separation": "review",
        "topic_coherence_note": "keep as sparse lexical baseline",
        "decision": "baseline",
    },
])

display(model_review_notes)


## S3 Topic Labeling Prompt Builder

Build a compact text payload for manual/LLM topic naming. 
For each topic we include the top 5 representative documents and 5 additional random documents from the same topic. 
The final block ends with a default mapping dictionary that can be edited in-place.


In [ ]:
TOP_REPRESENTATIVES_FOR_LABELING = 3
RANDOM_DOCS_FOR_LABELING = 7
LABELING_RANDOM_STATE = 0

s3_model = results["S3"]
s3_docs_for_labeling = s3_model.documents[[
    "topic_id",
    "topic_confidence",
    "question",
]].copy()


In [ ]:
def _build_topic_labeling_prompt(topic_model, top_k: int = 5, random_k: int = 5, random_state: int = 0) -> str:
    topic_summaries = topic_model.topic_summary().sort_values("topic_id").reset_index(drop=True)
    docs = topic_model.documents[["topic_id", "topic_confidence", "question"]].copy()

    lines = []
    lines.append("Choose short, human-readable names for the following S3 topics.")
    lines.append("Use the representative and random documents to infer a clean title for each topic.")
    lines.append("Prefer concrete event-family names over lexical keyword lists.")
    lines.append("")

    for row in topic_summaries.itertuples(index=False):
        topic_id = int(row.topic_id)
        topic_key = f"topic_{topic_id}"
        topic_docs = docs.loc[docs["topic_id"] == topic_id].copy()
        topic_docs = topic_docs.sort_values("topic_confidence", ascending=False).reset_index(drop=True)

        top_docs = topic_docs.head(top_k).copy()
        remaining = topic_docs.iloc[top_k:].copy()
        if len(remaining) > 0:
            random_docs = remaining.sample(n=min(random_k, len(remaining)), random_state=random_state)
        else:
            random_docs = remaining.head(0).copy()

        lines.append(f"{topic_key}")
        lines.append(f"Current label: {row.topic_label}")
        lines.append(f"Current terms: {row.topic_terms}")
        lines.append(f"Topic size: {int(row.topic_size)}")
        lines.append("Top 5 representative documents:")
        for idx, doc in enumerate(top_docs.itertuples(index=False), start=1):
            lines.append(f"  R{idx}. {doc.question}")
        lines.append("Random 5 additional documents:")
        for idx, doc in enumerate(random_docs.itertuples(index=False), start=1):
            lines.append(f"  X{idx}. {doc.question}")
        lines.append("")

    default_mapping = {f"topic_{int(row.topic_id)}": f"topic_{int(row.topic_id)}" for row in topic_summaries.itertuples(index=False)}
    lines.append("Return only a Python dict with one short title per topic, starting from this default skeleton:")
    lines.append(str(default_mapping))
    return "\n".join(lines)


s3_topic_labeling_prompt = _build_topic_labeling_prompt(
    s3_model,
    top_k=TOP_REPRESENTATIVES_FOR_LABELING,
    random_k=RANDOM_DOCS_FOR_LABELING,
    random_state=LABELING_RANDOM_STATE,
)

print(s3_topic_labeling_prompt)


## Topic Assignment Artifact

Build an export-ready market-to-topic table from the fitted S3 model. This is the main annotation artifact to join back onto canonical markets.


In [ ]:
ai_topic_names = {
    'topic_0': 'Crypto Price Targets',
    'topic_1': 'Year-End 2025 Superlatives',
    'topic_2': 'Election And 2025 Macro Outcomes',
    'topic_3': 'Speech Word-Usage Markets',
    'topic_4': 'Miscellaneous Political And Macro Events',
    'topic_5': 'Iran Strike Escalation',
    'topic_6': 'Opening Weekend Box Office',
    'topic_7': 'Trump Bilateral Meetings',
    'topic_8': 'Mayoral Elections',
    'topic_9': 'Mixed Policy And Resolution Markets',
    'topic_10': 'Miscellaneous Public-Event Outcomes',
    'topic_11': 'US-Venezuela Escalation',
    'topic_12': 'Recognition And Geopolitical Status',
    'topic_13': 'NFL And Big Game Markets',
    'topic_14': 'Russia Territorial Advances',
    'topic_15': 'International Football Matches',
    'topic_16': 'Streaming Charts',
    'topic_17': 'Mixed Politics And Elon Markets',
    'topic_18': 'Quarterly Earnings Beats',
    'topic_19': 'Trump Tariffs And Mixed Politics'
}

In [ ]:
ai_category_names = {
    'topic_0': 'Crypto',
    'topic_1': 'Year-End Rankings',
    'topic_2': 'Elections And Macro',
    'topic_3': 'Speech And Language',
    'topic_4': 'Politics And Macro',
    'topic_5': 'Geopolitics',
    'topic_6': 'Entertainment',
    'topic_7': 'Trump And Diplomacy',
    'topic_8': 'Elections',
    'topic_9': 'Policy And Resolution',
    'topic_10': 'Miscellaneous Public Events',
    'topic_11': 'Geopolitics',
    'topic_12': 'Geopolitical Recognition',
    'topic_13': 'Sports',
    'topic_14': 'Geopolitics',
    'topic_15': 'Sports',
    'topic_16': 'Streaming And Charts',
    'topic_17': 'Politics And Elon',
    'topic_18': 'Earnings',
    'topic_19': 'Trump Policy'
}


In [ ]:
s3_model = results["S3"]

s3_topic_assignments = s3_model.documents[[
    "market_id",
    "topic_id",
    "topic_confidence",
    "topic_label",
    "topic_terms",
]].copy()

s3_topic_assignments["topic_name"] = s3_topic_assignments["topic_id"].map(
    lambda value: ai_topic_names.get(f"topic_{int(value)}", f"topic_{int(value)}")
)
s3_topic_assignments["category_name"] = s3_topic_assignments["topic_id"].map(
    lambda value: ai_category_names.get(f"topic_{int(value)}", "Uncategorized")
)
s3_topic_assignments["topic_model_name"] = s3_model.model_name
s3_topic_assignments["topic_text_mode"] = s3_model.text_mode
s3_topic_assignments["topic_model_version"] = "s3_v1"

# Optional stable column ordering for downstream joins.
s3_topic_assignments = s3_topic_assignments[[
    "market_id",
    "topic_id",
    "topic_name",
    "category_name",
    "topic_confidence",
    "topic_label",
    "topic_terms",
    "topic_model_name",
    "topic_text_mode",
    "topic_model_version",
]]

display(s3_topic_assignments.head(10))


## Topic-To-Benchmark Features

Derive benchmark-facing features from the fitted S3 topic mixture. These are the compact signals most likely to be useful in forecasting or retrieval benchmarks.


In [ ]:
import numpy as np

s3_topic_matrix = pd.DataFrame(
    s3_model.doc_topic_matrix,
    index=s3_model.documents.index,
    columns=[f"topic_score_{int(topic_id)}" for topic_id in s3_model.topics["topic_id"]],
)

s3_topic_features = pd.concat(
    [
        s3_model.documents[["market_id", "topic_id", "topic_confidence"]]
        .assign(
            topic_name=lambda frame: frame["topic_id"].map(
                lambda value: ai_topic_names.get(f"topic_{int(value)}", f"topic_{int(value)}")
            ),
            category_name=lambda frame: frame["topic_id"].map(
                lambda value: ai_category_names.get(f"topic_{int(value)}", "Uncategorized")
            ),
        )
        .reset_index(drop=True),
        s3_topic_matrix.reset_index(drop=True),
    ],
    axis=1,
)

sorted_scores = np.sort(s3_model.doc_topic_matrix, axis=1)[:, ::-1]
s3_topic_features["topic_entropy"] = -np.sum(
    np.clip(s3_model.doc_topic_matrix, 1e-12, 1.0) * np.log(np.clip(s3_model.doc_topic_matrix, 1e-12, 1.0)),
    axis=1,
)
s3_topic_features["topic_top2_margin"] = sorted_scores[:, 0] - sorted_scores[:, 1]
s3_topic_features["topic_model_name"] = s3_model.model_name
s3_topic_features["topic_model_version"] = "s3_v1"

display(s3_topic_features.head(10))


## Export Cells

Write the main S3 artifacts to disk so they can be reused in benchmark code or paper analysis.


In [ ]:
TOPIC_ARTEFACT_DIR = REPO_ROOT / "research_notebooks" / "running_artefacts_new" / "topic_models" / "s3_v1"
TOPIC_ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

s3_topic_assignments_export = s3_topic_assignments.copy()
s3_topic_assignments_export["topic_name"] = s3_topic_assignments_export["topic_id"].map(lambda value: ai_topic_names.get(f"topic_{int(value)}", f"topic_{int(value)}"))
s3_topic_assignments_export.to_parquet(TOPIC_ARTEFACT_DIR / "topic_assignments.parquet", index=False)
s3_topic_features.to_parquet(TOPIC_ARTEFACT_DIR / "topic_features.parquet", index=False)
s3_topic_summary_export = s3_model.topic_summary().copy()
s3_topic_summary_export["topic_name"] = s3_topic_summary_export["topic_id"].map(lambda value: ai_topic_names.get(f"topic_{int(value)}", f"topic_{int(value)}"))
s3_topic_summary_export["category_name"] = s3_topic_summary_export["topic_id"].map(lambda value: ai_category_names.get(f"topic_{int(value)}", "Uncategorized"))
s3_topic_summary_export.to_parquet(TOPIC_ARTEFACT_DIR / "topic_summary.parquet", index=False)
comparison_summary.to_parquet(TOPIC_ARTEFACT_DIR / "model_comparison_summary.parquet", index=False)

ai_topic_name_mapping = pd.DataFrame(
    [
        {
            "topic_key": key,
            "topic_name": value,
            "category_name": ai_category_names.get(key, "Uncategorized"),
        }
        for key, value in ai_topic_names.items()
    ]
)
ai_topic_name_mapping.to_parquet(TOPIC_ARTEFACT_DIR / "topic_name_mapping.parquet", index=False)

artifact_metadata = pd.DataFrame([
    {
        "topic_model_name": s3_model.model_name,
        "topic_model_version": "s3_v1",
        "text_mode": s3_model.text_mode,
        "n_topics": len(s3_model.topics),
        "n_documents": len(s3_model.documents),
        "market_limit": MARKET_LIMIT,
        "market_order": MARKET_ORDER,
        "selected_by_visual_review": True,
    }
])
artifact_metadata.to_parquet(TOPIC_ARTEFACT_DIR / "artifact_metadata.parquet", index=False)

print("Saved topic artefacts to:", TOPIC_ARTEFACT_DIR)


In [ ]:
s3_topic_assignments

## Reload Exported Topic Artefacts

Example of how to load the exported topic artefacts back into a notebook or benchmark script.


In [ ]:
TOPIC_ARTEFACT_DIR = REPO_ROOT / "research_notebooks" / "running_artefacts_new" / "topic_models" / "s3_v1"

loaded_topic_assignments = pd.read_parquet(TOPIC_ARTEFACT_DIR / "topic_assignments.parquet")
loaded_topic_features = pd.read_parquet(TOPIC_ARTEFACT_DIR / "topic_features.parquet")
loaded_topic_summary = pd.read_parquet(TOPIC_ARTEFACT_DIR / "topic_summary.parquet")
loaded_topic_name_mapping = pd.read_parquet(TOPIC_ARTEFACT_DIR / "topic_name_mapping.parquet")
loaded_artifact_metadata = pd.read_parquet(TOPIC_ARTEFACT_DIR / "artifact_metadata.parquet")

display(loaded_topic_assignments.head(5))
display(loaded_topic_features.head(5))
display(loaded_topic_summary.head(5))


In [ ]:
question2topic = (
    canonical.markets[["market_id", "question"]]
    .merge(
        loaded_topic_assignments[["market_id", "topic_name", "category_name"]],
        on="market_id",
        how="left",
    )
    .reset_index(drop=True)
)
question2topic


In [ ]:

question2topic = s3_model.documents[["market_id", "question"]].merge(
    loaded_topic_assignments[["market_id", "topic_name", "category_name"]],
    on="market_id",
    how="left",
).reset_index(drop=True)


In [ ]:
question2topic[question2topic.category_name == 'Speech And Language'].sample(15)

In [ ]:
question2topic[question2topic.category_name == 'Sports'].sample(15)

In [ ]:
question2topic.category_name.value_counts()

In [ ]:
question2topic[question2topic.category_name == 'Crypto'].sample(15)